[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C61_Detection_Practice_Interview_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境（指标意识自测 / 归因信息增益 / run manifest 校验）

目标：把本课的三个核心能力**各做成一段可运行的代码**，而不是三句口号。
跑完这个 notebook，你会得到一份自己的「指标意识」得分，和三个可以直接搬进项目的小工具。

本 notebook 你会亲手实现：
1. **环境自检**：确认纯 numpy + 标准库环境可用，且**同种子同结果**（本课全部实验的前提）
2. **指标换算器**：TP/FP/FN → precision / recall / F1 / **FP per km**
   —— 并算出「precision 0.90 在车上意味着每公里 54 次误报」
3. **指标意识自测**：14 条真实改动，你先猜影响量级，再对照经验区间打分
4. **ROI 与帕累托前沿**：每个改动的收益/代价比，找出「零延迟代价的免费改进」
5. **归因的信息增益**：把「下一步该做哪个检查」变成一个可计算的量（$\mathrm{IG}=H_b(m)$）
6. **run manifest 校验器**：缺任何一个必需字段就拒绝这次 run

> 心智模型：**指标意识 = 改动前能给出区间；归因 = 知道下一个检查做什么；
> 交付纪律 = 任意历史状态可重建。三者都能写成代码。**

## 1 · 环境自检：本课不需要 GPU，但需要确定性

In [ ]:
import sys, math, json, platform
import numpy as np

print('python  :', sys.version.split()[0])
print('numpy   :', np.__version__)
print('platform:', platform.platform())
assert sys.version_info >= (3, 8), '需要 Python 3.8+'

# 本课全部实验的前提：**同种子同结果**。做不到这一点，后面所有对比都无意义。
a = np.random.default_rng(7).normal(size=5)
b = np.random.default_rng(7).normal(size=5)
c = np.random.default_rng(8).normal(size=5)
print('\nrng(7) 第一次:', np.round(a, 6))
print('rng(7) 第二次:', np.round(b, 6))
print('rng(8)       :', np.round(c, 6))
assert np.array_equal(a, b), '同种子必须完全相同'
assert not np.allclose(a, c), '不同种子必须不同（否则种子没起作用）'

rng = np.random.default_rng(20260817)   # 全 notebook 统一入口
print('\n✅ 环境就位：纯 numpy + 标准库，CPU 可跑，无需联网。')
print('   注意 np.random.default_rng(seed) 与全局 np.random.seed 是两套状态 ——')
print('   真实项目里必须同时固定：python random / numpy / torch / cudnn.deterministic。')

## 2 · 指标换算器：precision 0.90 在车上是什么意思

整体 precision / recall 是**离线**语言。车端能行动的语言是
**每公里误报次数（FP/km）** 与 **首次检出距离**。两者之间只差一个换算，
但换算完的数字常常让人重新认识自己的模型。

In [ ]:
def metrics_from_counts(tp, fp, fn, n_frames):
    '''最小指标闭包：从三个计数出发，得到所有常用量。'''
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return dict(precision=prec, recall=rec, f1=f1,
                miss_rate=1 - rec, fp_per_frame=fp / n_frames)

def frames_per_km(fps, speed_kmh):
    '''以 speed_kmh 行驶 1 km，相机吐出多少帧。'''
    return fps * 3600.0 / speed_kmh

def fp_per_km(fp_per_frame, fps=30, speed_kmh=100):
    return fp_per_frame * frames_per_km(fps, speed_kmh)

# 一个"看起来还行"的 TSR 评测结果
TP, FP, FN, N_FRAMES = 900, 100, 100, 2000
m = metrics_from_counts(TP, FP, FN, N_FRAMES)
print(f"TP={TP}  FP={FP}  FN={FN}  帧数={N_FRAMES}")
for k, v in m.items():
    print(f'  {k:<14s} {v:.4f}')
assert abs(m['precision'] - 0.9) < 1e-12 and abs(m['recall'] - 0.9) < 1e-12
assert abs(m['f1'] - 0.9) < 1e-12
assert abs(m['fp_per_frame'] - 0.05) < 1e-12

fpk = fp_per_km(m['fp_per_frame'], fps=30, speed_kmh=100)
print(f"\n30 FPS、100 km/h -> 每公里 {frames_per_km(30,100):.0f} 帧")
print(f"precision 0.90  ==>  **每公里 {fpk:.0f} 次误报**")
assert abs(frames_per_km(30, 100) - 1080.0) < 1e-9
assert abs(fpk - 54.0) < 1e-9
print('\n⚠️  离线看是"90% 精确率，还不错"；换算到车上是**每公里 54 次误报** —— 完全不可上线。')

In [ ]:
# 反过来问：要做到「每公里 <= 1 次误报」，precision 需要多少？
def precision_for_fp_budget(tp, n_frames, fp_per_km_budget, fps=30, speed_kmh=100):
    fpf_budget = fp_per_km_budget / frames_per_km(fps, speed_kmh)
    fp_allowed = fpf_budget * n_frames
    return tp / (tp + fp_allowed), fp_allowed

for budget in [10.0, 5.0, 1.0, 0.2]:
    p, fp_allowed = precision_for_fp_budget(TP, N_FRAMES, budget)
    print(f'FP/km <= {budget:>5.1f}  ->  评测集里最多 {fp_allowed:6.2f} 个 FP  ->  precision >= {p:.5f}')

p1, fp1 = precision_for_fp_budget(TP, N_FRAMES, 1.0)
assert abs(fp1 - 2000 / 1080) < 1e-9, fp1          # 1/1080 * 2000
assert p1 > 0.997, p1
print(f'\n✅ 目标 1 FP/km 对应 precision >= {p1:.4f} —— 从 0.90 到 0.998，')
print('   FP 要从 100 降到 1.85 个，是 **54 倍**的差距，不是"再调调阈值"能到的。')
print('\n📌 TSR 落点：这就是为什么量产 TSR 必须用**时序多帧确认 + 迟滞**（C55 m04）——')
print('   单帧 precision 到不了 0.998，但"连续 3 帧一致才上报"能把独立误报压掉 2 个数量级。')

## 3 · 「指标意识」自测

下面是 14 条真实的检测改动。**先自己在心里给出一个数**（对哪个指标、涨/掉多少），
再运行打分 cell 对照经验区间。

区间是「经验量级」不是「定律」：它依赖数据分布、训练预算、模型容量。
**报区间而不报点估计，本身就是指标意识的一部分。**

In [ ]:
# 经验量级表（教学用；量级与公开论文/工程实践一致，不是精确复现某一篇）
# delta_lo / delta_hi：对"目标指标"的影响区间（单位：AP 点）
# dlat_ms：相对 baseline(9.0 ms) 的延迟增量
CHANGES = [
    dict(id='res',      name='输入分辨率 640 -> 1280',              metric='mAP_small', lo= 3.0, hi= 6.0,  dlat_ms=22.5, conf='高'),
    dict(id='p2',       name='FPN 加 P2 层 (stride 4)',             metric='mAP_small', lo= 1.5, hi= 4.0,  dlat_ms= 3.0, conf='高'),
    dict(id='mosaic',   name='Mosaic + 末 10 epoch 关闭',           metric='mAP',       lo= 0.8, hi= 2.0,  dlat_ms= 0.0, conf='高'),
    dict(id='cp',       name='稀有类 copy-paste',                   metric='尾部类 AP', lo= 3.0, hi=15.0,  dlat_ms= 0.0, conf='中'),
    dict(id='backbone', name='backbone R50 -> R101',                metric='mAP',       lo= 1.0, hi= 2.0,  dlat_ms= 3.6, conf='高'),
    dict(id='epochs',   name='训练 12 -> 36 epoch',                 metric='mAP',       lo= 1.5, hi= 3.0,  dlat_ms= 0.0, conf='高'),
    dict(id='assign',   name='标签分配 MaxIoU -> TaskAligned',      metric='mAP',       lo= 1.0, hi= 2.5,  dlat_ms= 0.0, conf='高'),
    dict(id='nwd',      name='小目标度量 IoU -> NWD',               metric='mAP_small', lo= 1.0, hi= 3.0,  dlat_ms= 0.0, conf='中'),
    dict(id='nmsthr',   name='NMS IoU 阈值 0.50 -> 0.45',           metric='mAP',       lo=-0.2, hi= 0.2,  dlat_ms= 0.0, conf='低'),
    dict(id='gamma',    name='Focal gamma 2.0 -> 1.5',              metric='mAP',       lo=-0.3, hi= 0.3,  dlat_ms= 0.0, conf='低'),
    dict(id='int8ok',   name='INT8 PTQ（校准集覆盖长尾场景）',      metric='mAP',       lo=-1.0, hi=-0.3,  dlat_ms=-5.0, conf='高'),
    dict(id='int8bad',  name='INT8 PTQ（校准集全是白天晴天）',      metric='夜间桶 AP', lo=-15.0,hi=-5.0,  dlat_ms=-5.0, conf='高'),
    dict(id='hflip',    name='水平翻转增强（含左转/右转标志）',      metric='方向类 AP', lo=-20.0,hi=-5.0,  dlat_ms= 0.0, conf='高'),
    dict(id='tta',      name='TTA：多尺度 + 翻转 + WBF 融合',        metric='mAP',       lo= 0.5, hi= 1.5,  dlat_ms=27.0, conf='高'),
]
BASE_LAT = 9.0

def span_str(c):
    return '[%+.1f, %+.1f]' % (c['lo'], c['hi'])

print(f"{'改动':<34s} {'目标指标':<12s} {'经验区间(AP)':>16s} {'Δ延迟ms':>9s} {'置信'}")
for c in CHANGES:
    print(f"{c['name']:<34s} {c['metric']:<12s} {span_str(c):>16s} "
          f"{c['dlat_ms']:>9.1f} {c['conf']:>4s}")

n_free_lat = sum(1 for c in CHANGES if c['dlat_ms'] <= 0)
assert len(CHANGES) == 14
assert n_free_lat == 10, n_free_lat
print(f"\n✅ 14 条改动里，延迟代价 <= 0 的有 **{n_free_lat}** 条；")
print('   其中收益为正的才是真正的「免费改进」（下一节算出来是 5 条）。')
print('⚠️  注意 nmsthr 与 gamma 的区间是 [-0.2,+0.2] / [-0.3,+0.3] —— **跨零**，')
print('    意思是「这类改动的效果与种子噪声同量级」。模块 01 会把这句话变成一个检验。')

In [ ]:
def grade_guess(guess, lo, hi):
    '''打分规则（在**幅度空间**里放宽，这样正负区间对称处理）：
       命中   —— guess 落在经验区间内
       量级对 —— 方向一致，且幅度在 [0.5*min|区间|, 2*max|区间|] 内
       量级错 —— 其余（含方向反了）'''
    if lo <= guess <= hi:
        return '命中'
    if guess * (lo + hi) <= 0:                 # 方向不一致
        return '量级错'
    g = abs(guess)
    lo_m, hi_m = min(abs(lo), abs(hi)), max(abs(lo), abs(hi))
    return '量级对' if 0.5 * lo_m <= g <= 2.0 * hi_m else '量级错'

# 手算校验
assert grade_guess(2.0, 1.5, 4.0) == '命中'
assert grade_guess(5.0, 1.5, 4.0) == '量级对'      # 幅度 5 在 [0.75, 8] 内，方向同号
assert grade_guess(9.0, 1.5, 4.0) == '量级错'      # 幅度超过 2*4.0=8
assert grade_guess(-1.0, 1.5, 4.0) == '量级错'     # 方向反了
assert grade_guess(-0.4, -1.0, -0.3) == '命中'
assert grade_guess(-2.0, -15.0, -5.0) == '量级错'  # 方向对但幅度只有下界的 0.4 倍

# —— 一份"考生答卷"（把这里换成你自己的猜测，重跑本 cell）——
MY_GUESS = {'res': 4.0, 'p2': 2.5, 'mosaic': 1.2, 'cp': 8.0, 'backbone': 1.5,
            'epochs': 2.0, 'assign': 1.8, 'nwd': 2.0, 'nmsthr': 0.0, 'gamma': 0.1,
            'int8ok': -0.5, 'int8bad': -2.0, 'hflip': -1.0, 'tta': 1.0}

score = {'命中': 0, '量级对': 0, '量级错': 0}
print(f"{'改动':<34s} {'你的猜测':>9s} {'经验区间':>16s}  判定")
for c in CHANGES:
    g = MY_GUESS[c['id']]
    r = grade_guess(g, c['lo'], c['hi'])
    score[r] += 1
    print(f"{c['name']:<34s} {g:>+9.1f} {span_str(c):>16s}  {r}")
total = 2 * score['命中'] + 1 * score['量级对']
print(f"\n得分 {total} / {2*len(CHANGES)}   命中 {score['命中']} · 量级对 {score['量级对']} · 量级错 {score['量级错']}")
assert score['命中'] + score['量级对'] + score['量级错'] == 14
assert score['量级错'] >= 2, '这份示例答卷刻意错了 int8bad 与 hflip 两项'
print('\n⚠️  示例答卷错在 **int8bad(-2.0)** 与 **hflip(-1.0)**：都低估了「分桶指标」的剧烈程度。')
print('    这正是最常见的指标意识缺口 —— 用整体 mAP 的直觉去猜分桶指标，')
print('    而分桶指标的样本少、场景集中，波动幅度大一个数量级。')

## 4 · ROI 与免费改进：把代价写下来

$$\mathrm{ROI} = \frac{\Delta \mathrm{AP}}{\Delta T / T_0}, \qquad
\text{「必做项」} \iff \Delta T \le 0 \;\wedge\; \Delta \mathrm{AP} > 0$$

In [ ]:
def mid(c):  return 0.5 * (c['lo'] + c['hi'])

def roi(c, base_lat=BASE_LAT):
    '''收益/代价比。延迟代价 <= 0 的改动没有分母 -> 记为 inf（免费改进）。'''
    d = c['dlat_ms'] / base_lat
    if d <= 0:
        return float('inf') if mid(c) > 0 else float('-inf')
    return mid(c) / d

free_wins = [c for c in CHANGES if c['dlat_ms'] <= 0 and mid(c) > 0]
costly    = [c for c in CHANGES if c['dlat_ms'] > 0]

print('【免费改进】ΔT <= 0 且 ΔAP > 0 —— 不消耗延迟预算，应无条件先做')
for c in sorted(free_wins, key=mid, reverse=True):
    print(f"  {c['name']:<34s} {mid(c):+6.2f} AP  ({c['metric']})")
assert {c['id'] for c in free_wins} == {'mosaic', 'cp', 'epochs', 'assign', 'nwd'}, \
    [c['id'] for c in free_wins]

print('\n【要花延迟预算的改动】按 ROI 排序')
print(f"  {'改动':<34s} {'ΔAP':>7s} {'Δ延迟':>8s} {'相对延迟':>9s} {'ROI':>8s}")
for c in sorted(costly, key=roi, reverse=True):
    print(f"  {c['name']:<34s} {mid(c):>+7.2f} {c['dlat_ms']:>+8.1f} "
          f"{c['dlat_ms']/BASE_LAT:>8.1%} {roi(c):>8.2f}")

assert roi(next(c for c in CHANGES if c['id'] == 'p2')) > \
       roi(next(c for c in CHANGES if c['id'] == 'tta')), 'P2 的 ROI 应远高于 TTA'
print('\n✅ TTA 的 ROI 垫底（+1.0 AP 换 3 倍延迟）—— **车端基本不可用**，')
print('   但它在离线打标 / 生成伪标签时非常有用。同一个改动，语境不同结论相反。')
print('📌 面试点：被问「你会怎么提升 TSR 的 mAP」时，**先把免费改进列完再谈架构**，')
print('   这个顺序本身就是「知道延迟是预算」的信号。')

## 5 · 归因：把「下一步查什么」变成可计算的量

**关键等式**：一个把候选原因按概率质量 $m$ / $1-m$ 切成两块的二值检查，
其信息增益恰好等于 $H_b(m)$ —— 与候选集内部的分布无关。
所以**最优检查 = 把质量切得最接近 50/50 的那一个**。

In [ ]:
def entropy(ps):
    ps = [p for p in ps if p > 0]
    return -sum(p * math.log2(p) for p in ps)

def h_binary(m):
    if m <= 0 or m >= 1:
        return 0.0
    return -m * math.log2(m) - (1 - m) * math.log2(1 - m)

# 症状：mAP 恒为 0，但 loss 正常下降。候选原因与先验（来自历史事故统计）
CAUSES = {
    '类别 ID 偏移 0/1':          0.35,
    '坐标格式弄反 xywh<->xyxy':  0.25,
    '评测集与训练集类别表不一致': 0.15,
    '学习率过大导致框全发散':     0.10,
    '标注坐标未随 resize 缩放':   0.10,
    '数据没打乱（每 batch 单类）': 0.05,
}
assert abs(sum(CAUSES.values()) - 1.0) < 1e-12

# 每个检查 -> 它为"真"时能确认的原因子集
CHECKS = {
    'A. 把 GT 当预测送进评测器，看 mAP 是否 = 1.0':
        {'类别 ID 偏移 0/1', '坐标格式弄反 xywh<->xyxy',
         '评测集与训练集类别表不一致', '标注坐标未随 resize 缩放'},
    'B. 打印前 20 个预测框，看坐标范围是 [0,1] 还是像素':
        {'坐标格式弄反 xywh<->xyxy', '标注坐标未随 resize 缩放'},
    'C. 看 loss 曲线 3 个 epoch 后是否仍在降':
        {'学习率过大导致框全发散', '数据没打乱（每 batch 单类）'},
    'D. 统计一个 batch 里的类别数是否 > 1':
        {'数据没打乱（每 batch 单类）'},
    'E. 打印训练与评测两侧的 class_id -> name 映射表':
        {'类别 ID 偏移 0/1', '评测集与训练集类别表不一致'},
}

def info_gain(causes, subset):
    m = sum(p for k, p in causes.items() if k in subset)
    return m, h_binary(m)

H0 = entropy(list(CAUSES.values()))
print(f'先验熵 H0 = {H0:.4f} bit（{len(CAUSES)} 个候选，均匀时是 {math.log2(len(CAUSES)):.4f}）\n')
print(f"{'检查':<48s} {'切出质量 m':>11s} {'IG (bit)':>10s}")
rows = []
for name, sub in CHECKS.items():
    m_, ig = info_gain(CAUSES, sub)
    rows.append((ig, m_, name))
    print(f'{name:<48s} {m_:>11.2f} {ig:>10.4f}')

best_ig, best_m, best_name = max(rows)
# 与"最接近 50/50"的检查一致 —— 这是 IG = H_b(m) 的直接推论
closest = min(rows, key=lambda r: abs(r[1] - 0.5))
assert best_name == closest[2], (best_name, closest[2])
assert abs(best_ig - h_binary(best_m)) < 1e-12
assert best_name.startswith('E'), best_name
print(f'\n▶ 最优首查：{best_name}（m={best_m:.2f}, IG={best_ig:.4f} bit）')
print(f'▶ 直觉上"最全面"的检查 A 切出 m=0.85，IG 只有 {h_binary(0.85):.4f} bit ——')
print('  它几乎总是给同一个答案，做完候选集还是那么大。')
print('\n✅ 心法：**检查的价值由判别性决定，不由覆盖面决定**（和二分查找是同一个道理）。')

## 6 · 交付纪律：run manifest 校验器

规则很简单：**缺任何一个必需字段，这次 run 就不算数**。
把它放进训练脚本的最后一步，「存在一个 manifest」就等价于「这次实验可复现」。

In [ ]:
REQUIRED = [
    ('run_id',       lambda v: isinstance(v, str) and len(v) > 0),
    ('code.commit',  lambda v: isinstance(v, str) and len(v) >= 7),
    ('code.dirty',   lambda v: v is False),              # ← True 直接判不可复现
    ('data.train',   lambda v: isinstance(v, str) and len(v) > 0),
    ('data.val',     lambda v: isinstance(v, str) and len(v) > 0),
    ('config_hash',  lambda v: isinstance(v, str) and len(v) >= 6),
    ('env.python',   lambda v: isinstance(v, str)),
    ('env.gpu',      lambda v: isinstance(v, str)),
    ('seed',         lambda v: isinstance(v, int)),      # None 不行
    ('metrics.mAP',  lambda v: isinstance(v, (int, float))),
    ('metrics.by_size',  lambda v: isinstance(v, dict) and len(v) >= 3),   # 分桶必须有
    ('metrics.by_light', lambda v: isinstance(v, dict) and len(v) >= 2),
]

def dig(d, dotted):
    cur = d
    for part in dotted.split('.'):
        if not isinstance(cur, dict) or part not in cur:
            return None, False
        cur = cur[part]
    return cur, True

def validate_manifest(mf):
    '''返回 (ok, 问题列表)。问题分两类：missing（缺字段）/ invalid（有但不合规）。'''
    problems = []
    for key, rule in REQUIRED:
        val, present = dig(mf, key)
        if not present:
            problems.append(('missing', key))
        elif not rule(val):
            problems.append(('invalid', f'{key}={val!r}'))
    return (len(problems) == 0), problems

GOOD = {
    'run_id': '2026-08-17_tsr_p2head_s0',
    'code': {'commit': 'd41d8cd98f00b204', 'dirty': False, 'repo': 'perception/tsr'},
    'data': {'train': 'tsr_v7.2', 'val': 'tsr_eval_v3 (frozen 2026-06-01)'},
    'config_hash': 'a3f5c1e9',
    'env': {'python': '3.11.9', 'numpy': '1.26.4', 'gpu': 'A100-80G', 'driver': '550.54'},
    'seed': 0,
    'metrics': {'mAP': 0.8213, 'mAP50': 0.9410,
                'by_size': {'<16px': 0.412, '16-32': 0.701, '32-64': 0.868, '>64': 0.912},
                'by_light': {'day': 0.851, 'night': 0.674, 'backlit': 0.612},
                'fp_per_km': 0.83, 'latency_p99_ms': 9.2},
    'artifacts': {'ckpt': 's3://.../best.pth'},
}
BAD = json.loads(json.dumps(GOOD))          # 深拷贝后制造三个典型问题
BAD['code']['dirty'] = True                 # ① 工作区脏 -> 不可复现
BAD['seed'] = None                          # ② 没记种子 -> 无法做对照
del BAD['metrics']['by_light']              # ③ 只存了整体指标 -> 分桶对比永久失效

for label, mf in [('GOOD', GOOD), ('BAD', BAD)]:
    ok, probs = validate_manifest(mf)
    print(f'{label}: {"✅ 通过" if ok else "❌ 拒绝"}')
    for kind, detail in probs:
        print(f'    [{kind}] {detail}')

ok_g, _ = validate_manifest(GOOD)
ok_b, probs_b = validate_manifest(BAD)
assert ok_g, '完整 manifest 应通过'
assert not ok_b and len(probs_b) == 3, probs_b
assert ('invalid', 'code.dirty=True') in probs_b
assert ('missing', 'metrics.by_light') in probs_b
print('\n✅ 三个问题各自对应一类事故：')
print('   dirty=True   -> 这份代码在世界上任何地方都不存在，永久不可复现')
print('   seed=None    -> 无法判断后续对比里的差异是改动还是噪声（m01 全模块的主题）')
print('   缺 by_light  -> 存储成本几 KB，重建成本是重跑一次评测（如果还建得出来）')

## ✏️ 练习 1：FP/km 与工作点反推

实现两个函数：
- `fp_per_km_from_precision(tp, fp, n_frames, fps, speed_kmh)` → 每公里误报次数
- `min_precision_for(tp, n_frames, budget_fp_km, fps, speed_kmh)` → 达到该 FP/km 预算所需的最小 precision

提示：`frames_per_km = fps * 3600 / speed_kmh`；`fp_allowed = budget/frames_per_km * n_frames`。

In [ ]:
def fp_per_km_from_precision(tp, fp, n_frames, fps=30, speed_kmh=100):
    # TODO
    raise NotImplementedError

def min_precision_for(tp, n_frames, budget_fp_km, fps=30, speed_kmh=100):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(fp_per_km_from_precision(900, 100, 2000) - 54.0) < 1e-9
assert abs(fp_per_km_from_precision(900, 0, 2000) - 0.0) < 1e-12
# 车速减半 -> 每公里帧数翻倍 -> FP/km 翻倍
assert abs(fp_per_km_from_precision(900, 100, 2000, speed_kmh=50) - 108.0) < 1e-9
p = min_precision_for(900, 2000, 1.0)
assert abs(p - 900 / (900 + 2000 / 1080)) < 1e-12, p
assert min_precision_for(900, 2000, 0.2) > min_precision_for(900, 2000, 5.0)
for b in [10, 1, 0.2]:
    print(f'FP/km <= {b:>5}: precision >= {min_precision_for(900, 2000, b):.5f}')
print('✅ 练习 1 通过：**离线 precision 与车端 FP/km 之间差一个 1080 倍的换算**，')
print('   不做这个换算，就永远不知道自己离可上线有多远。')

## ✏️ 练习 2：帕累托前沿

实现 `pareto_front(items)`：`items` 是 `[(名字, 收益, 代价), ...]`。
若存在另一项 **收益 >= 且 代价 <=**，且至少一项严格更优，则本项被**支配**，应剔除。
返回未被支配的项，按代价升序。

In [ ]:
def pareto_front(items):
    # TODO: items = [(name, gain, cost), ...] -> 未被支配的子集，按 cost 升序
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测（手算）——
demo = [('a', 2.0, 1.0), ('b', 1.0, 1.0), ('c', 3.0, 5.0), ('d', 2.0, 3.0), ('e', 2.0, 1.0)]
front = pareto_front(demo)
names = sorted(n for n, _, _ in front)
# b 被 a 支配（同代价更低收益）；d 被 a 支配（同收益更高代价）
# a 与 e 完全相同 -> 互不严格支配，两者都保留
assert names == ['a', 'c', 'e'], names
assert [c for _, _, c in front] == sorted(c for _, _, c in front), '要按 cost 升序'
assert pareto_front([]) == [] or list(pareto_front([])) == []

real = [(c['name'], mid(c), c['dlat_ms']) for c in CHANGES if c['dlat_ms'] > 0]
for n, g, c_ in pareto_front(real):
    print(f'{n:<34s} 收益 {g:+5.2f} AP   代价 {c_:+5.1f} ms')
assert any('P2' in n for n, _, _ in pareto_front(real)), 'P2 应在前沿上'
print('✅ 练习 2 通过：前沿之外的改动**没有讨论价值** —— 存在一个各方面都不差的替代。')

## ✏️ 练习 3：最优首查

实现 `best_check(causes, checks)`：返回 `(检查名, 切出质量 m, 信息增益)`，
取信息增益最大者（并列时取名字字典序最小的，保证结果确定）。
用 `IG = H_b(m)`，其中 `m` 是该检查覆盖的原因的**概率质量之和**。

In [ ]:
def best_check(causes, checks):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
name, m_, ig = best_check(CAUSES, CHECKS)
assert name.startswith('E'), name
assert abs(m_ - 0.50) < 1e-12, m_          # 0.35 + 0.15
assert abs(ig - 1.0) < 1e-12, ig           # H_b(0.5) = 1 bit，二值检查的上限
# 极端情形：检查覆盖全部原因 -> 零信息
allc = {'X': set(CAUSES)}
assert abs(best_check(CAUSES, allc)[2] - 0.0) < 1e-12
# 并列时取字典序最小
tie = {'B_check': {'类别 ID 偏移 0/1', '评测集与训练集类别表不一致'},
       'A_check': {'坐标格式弄反 xywh<->xyxy', '学习率过大导致框全发散',
                   '标注坐标未随 resize 缩放', '数据没打乱（每 batch 单类）'}}
assert best_check(CAUSES, tie)[0] == 'A_check', best_check(CAUSES, tie)
print(f'最优首查: {name}\n  m = {m_:.2f}, IG = {ig:.4f} bit（二值检查的理论上限）')
print('✅ 练习 3 通过：**一次检查最多消掉 1 bit**，所以 6 个候选至少要 log2(6)=2.58 -> 3 次。')
print('   如果你的排查平均要 6 步，说明每一步都没在切一半。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def fp_per_km_from_precision(tp, fp, n_frames, fps=30, speed_kmh=100):
    return fp / n_frames * (fps * 3600.0 / speed_kmh)

def min_precision_for(tp, n_frames, budget_fp_km, fps=30, speed_kmh=100):
    fp_allowed = budget_fp_km / (fps * 3600.0 / speed_kmh) * n_frames
    return tp / (tp + fp_allowed)

In [ ]:
# 练习 2 参考答案
def pareto_front(items):
    out = []
    for i, (n, g, c) in enumerate(items):
        dominated = any(
            (g2 >= g and c2 <= c) and (g2 > g or c2 < c)
            for j, (_, g2, c2) in enumerate(items) if j != i
        )
        if not dominated:
            out.append((n, g, c))
    return sorted(out, key=lambda t: t[2])

In [ ]:
# 练习 3 参考答案
def best_check(causes, checks):
    best = None
    for name in sorted(checks):
        m = sum(p for k, p in causes.items() if k in checks[name])
        ig = h_binary(m)
        if best is None or ig > best[2] + 1e-15:
            best = (name, m, ig)
    return best

---
## 🧪 真实工程胶囊：接手一个检测模块的第一周

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 接手一个检测模块的第一周 —— 做完这 7 件事，你才算"能负责"
# （顺序是刻意的：先建立可观测性，再动模型）
# ══════════════════════════════════════════════════════════════════════

# ① 冻结一个评测集，并给它一个版本号
#    - 从此这个集合**只增不改**；改了就是新版本，历史数字全部作废
#    - 单独存一份 frozen 副本（不要只存一个 git 分支名）
#    $ sha256sum eval_v3.json > eval_v3.sha256
#    通过条件：任何人任何时间跑同一个 ckpt，得到完全相同的 mAP

# ② 把评测输出从"一个 mAP"改成"一张分桶表"
#    维度至少四个（来自 C55 失效模式 + C57 尺寸分层）：
#      size   : <16px / 16-32 / 32-64 / >64          ← TSR 的第一诊断维度
#      light  : day / night / backlit / tunnel
#      weather: clear / rain / fog / snow
#      class  : 全类别 AP + 关键类单列（stop / yield / speed_limit_*）
#    再加两个工程指标：fp_per_km、latency_p50/p99
#    通过条件：任意一次评测，能在 10 秒内回答"掉点掉在哪个桶"

# ③ 跑 baseline 的**多种子**（至少 3 个），把种子方差量出来
#    for s in 0 1 2; do train.py --seed $s --tag base_s$s; done
#    记录每个桶的 std。**这个数字是你之后所有"提升"的判据**（模块 01）
#    通过条件：你能说出"本项目 mAP 的种子标准差是 0.XX，所以 <0.YY 的差异不予采信"

# ④ 给每次 run 落 manifest（本 notebook 第 6 节的校验器）
#    训练脚本最后一步：validate_manifest(mf) 不过就不写产物
#    通过条件：随机抽一个 30 天前的 run，能从 manifest 重跑并落回方差区间

# ⑤ 建回归门禁（先建规则，再谈提升）
#    规则示例：新版本相对当前线上版本
#      · 任何桶的 AP 掉幅 > 2*std          -> 阻塞
#      · 关键类（stop/yield）掉任何点      -> 阻塞
#      · fp_per_km 上升 > 10%              -> 阻塞
#      · latency_p99 > budget              -> 阻塞
#    通过条件：门禁能自动跑，且**至少拦下过一次**（没拦过说明阈值太松）

# ⑥ 建 badcase 的分层抽样（不要"随便看几张"）
#    按 (桶 × 错误类型) 分层，每格抽 N 张，人工过一遍
#    通过条件：你能列出 top-5 失效模式，并给每个估计"修好能涨多少"（模块 02）

# ⑦ 演练一次回滚
#    随机挑一个历史版本，计时看多久能切回线上
#    通过条件：< 30 分钟，且不需要重新构建 engine

# ── 反模式（看到这些就说明前面某一步没做）────────────────────────────
#  · "这次涨了 0.4"          -> ③ 没做，不知道 0.4 是不是噪声
#  · "不知道为什么掉点了"     -> ② 没做，看不到分桶
#  · "上个季度那版怎么训的？" -> ④ 没做
#  · "先上了再说，掉点再回滚" -> ⑤⑦ 没做，回滚其实回不去
'''
print(RECIPE)
for token in ['冻结一个评测集', '分桶表', '多种子', 'manifest', '回归门禁',
              '分层抽样', '演练一次回滚', 'fp_per_km', 'latency_p99']:
    assert token in RECIPE, token
print('✅ 第一周检查单覆盖：可观测性(①②) -> 判据(③) -> 可复现(④) -> 门禁(⑤) -> 归因(⑥) -> 可回滚(⑦)')

### 小结

- **「会跑通 → 能改进 → 能负责」的差距不是知识量**，是三个能力：
  **指标意识**（改动前能给出区间与代价）、**归因能力**（指标动了能定位）、
  **交付纪律**（可复现 / 可回滚 / 可交接）。
- **整体 mAP 是最不敏感的指标**。加 P2 在 mAP_small 上 +1.5~4.0，稀释到整体只剩 +0.6~1.6。
  被问「涨了多少」时，**正确答法是「涨在哪」**。
- **离线 precision 与车端 FP/km 差一个 1080 倍的换算**（30 FPS、100 km/h）：
  precision 0.90 = **每公里 54 次误报**；要压到 1 FP/km 需要 precision 0.998。
  单帧做不到，所以量产 TSR 必须靠**时序确认 + 迟滞**。
- **把改动分成「免费的」和「要花延迟预算的」两类**。14 条改动里有 5 条延迟代价 ≤ 0
  （Mosaic / copy-paste / 延长训练 / 换标签分配 / 换小目标度量）——它们是帕累托改进，
  应无条件先做。TTA 的 ROI 垫底（+1.0 AP 换 3 倍延迟），车端不可用。
- **归因 = 挑判别性最强的检查，不是挑覆盖面最大的**。二值检查的信息增益
  $\mathrm{IG}=H_b(m)$ 只由它切出的概率质量决定，上限 1 bit ——
  所以 6 个候选原因至少需要 3 次检查，平均要 6 步说明每步都没在切一半。
- **manifest 是可复现的最小实现**：commit + dirty + 数据版本 + 配置 + 环境 + 种子 + **全量指标**。
  `dirty=True` 是红线；`metrics` 只存 mAP 会让历史分桶对比永久失效。

下一站：**模块 01 · 实验设计与归因** —— 种子方差有多大、+0.3 到底算不算提升、
以及「需要多少个种子才能检出它」。